# Investigating the Field Cancerisation Hypothesis

**Can a model detect whether normal-looking tissue came from a slide that also contains a tumor?**

---

## Background

This notebook systematically tests whether different model architectures can distinguish:
- **Class 0**: Normal tissue from normal slides (no tumor anywhere on the slide)
- **Class 1**: Normal-looking tissue from tumor slides (tumor exists elsewhere on the slide, but not in this patch)

If a model can reliably distinguish these classes, it suggests that normal tissue adjacent to tumors carries detectable changes. This could reflect:
- **Field cancerisation**: molecular/epigenetic changes in tissue surrounding tumors
- **Stromal response**: altered fibroblasts or extracellular matrix
- **Immune infiltration**: inflammatory cells responding to nearby tumor
- **Slide-level confounds**: staining differences, scanner artifacts (the null hypothesis)

## Experimental Design

We test multiple architectures of increasing complexity:

| Architecture | Description | Trainable Params |
|--------------|-------------|------------------|
| `subtle` | Custom CNN with fine-grained features | ~390K |
| `attention` | CNN with spatial attention mechanism | ~390K |
| `transfer` | Frozen MobileNetV2 (ImageNet features) | ~1.3K |
| `transfer_finetune` | MobileNetV2 with last 30 layers unfrozen | ~700K |

We also experiment with:
- Different learning rates
- Increased training epochs
- Data augmentation strategies

The key question: **Does any architecture achieve test AUC significantly above 0.5?**

In [ ]:
# === SECTION 1: SET UP THE PROJECT CODE ===

from pathlib import Path
import os

REPO_URL = 'https://github.com/balintstewart77/camelyon16-pathology.git'
REPO_DIR = Path('/content/camelyon16-pathology')

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/camelyon16-pathology
else:
    print("Repository already present in runtime.")

%cd /content/camelyon16-pathology

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

print("Project environment ready.")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

np.random.seed(42)

print("Setup complete!")


In [ ]:
# Configuration
from config import DEFAULT_CONFIG
from src.models import run_binary_experiment
from src.models.architectures import get_model, MODEL_REGISTRY

# === SECTION 2: MOUNT GOOGLE DRIVE AND LOCATE THE DATASET ===

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# The notebooks will check these locations in order:
# 1. Your original Drive path: /content/drive/MyDrive/new_work/Projects/pathovis_project/data
# 2. A shortcut in My Drive pointing to the shared folder below:
#    https://drive.google.com/drive/folders/1Ny8zXjBPsrqSFUD61v02EHsjAwgGSY47?usp=drive_link
#    If you use the shortcut option, place it directly under "My Drive"
#    and name it exactly: camelyon16_data

candidate_roots = [
    Path('/content/drive/MyDrive/new_work/Projects/pathovis_project/data'),
    Path('/content/drive/MyDrive/camelyon16_data'),
]

DATA_ROOT = next((root for root in candidate_roots if root.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find a CAMELYON16 dataset root.\n\n"
        "Checked these locations:\n"
        + "\n".join(f"  - {root}" for root in candidate_roots)
        + "\n\n"
        "To fix this, either:\n"
        "  1. Use the original path under new_work/Projects/pathovis_project/data\n"
        "  2. Or add this shared folder to Drive as a shortcut:\n"
        "     https://drive.google.com/drive/folders/1Ny8zXjBPsrqSFUD61v02EHsjAwgGSY47?usp=drive_link\n"
        "     Place it directly under 'My Drive' and name it exactly: camelyon16_data\n"
    )
TRAIN_PATH = DATA_ROOT / 'camelyon16_4class_stain_normalised'
TEST_PATH = DATA_ROOT / 'camelyon16_test_stain_normalised'

for path, name in [(TRAIN_PATH, 'Training dataset'), (TEST_PATH, 'Test dataset')]:
    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found at:\n  {path}\n\n"
            "Dataset root found at:\n"
            f"  {DATA_ROOT}\n\n"
            "Expected subfolders:\n"
            "  - camelyon16_4class_stain_normalised\n"
            "  - camelyon16_test_stain_normalised"
        )

print("Dataset paths verified.")
print(f"TRAIN_PATH = {TRAIN_PATH}")
print(f"TEST_PATH  = {TEST_PATH}")

print(f"Available architectures: {list(MODEL_REGISTRY.keys())}")
print(f"Training data: {TRAIN_PATH}")
print(f"Test data: {TEST_PATH}")


In [ ]:
# Training configuration
DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

# Store results for comparison
results_summary = []

## Experiment 1: Subtle CNN (Baseline)

Our custom CNN designed for detecting subtle contextual differences. This is the same architecture used in the main blog notebook.

In [ ]:
# Inspect the subtle model architecture
model = get_model('subtle')
model.summary()

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: Subtle CNN (baseline)")
print("=" * 60)

exp1_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,  # Slide context detection (class 0 vs class 1)
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

results_summary.append({
    'architecture': 'subtle',
    'learning_rate': 1e-5,
    'epochs': 15,
    'val_auc': exp1_results['results']['auc'],
    'val_acc': exp1_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp1_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp1_results['results']['accuracy']:.1%}")

## Experiment 2: Attention Model

Adds a spatial attention mechanism to focus on the most discriminative regions. If field cancerisation effects are localised (e.g., near vessels or in specific tissue structures), attention may help.

In [ ]:
# Inspect the attention model architecture
model = get_model('attention')
model.summary()

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Attention Model")
print("=" * 60)

exp2_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='attention',
    epochs=15,
    learning_rate=1e-5
)

results_summary.append({
    'architecture': 'attention',
    'learning_rate': 1e-5,
    'epochs': 15,
    'val_auc': exp2_results['results']['auc'],
    'val_acc': exp2_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp2_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp2_results['results']['accuracy']:.1%}")

## Experiment 3: Frozen Transfer Learning (MobileNetV2)

Uses ImageNet-pretrained MobileNetV2 as a fixed feature extractor. Only the final classification layer is trained.

**Hypothesis**: If ImageNet features (edges, textures, shapes) can detect field cancerisation, this should work. If it requires histopathology-specific features, this will fail.

In [ ]:
# Inspect the transfer model architecture
model = get_model('transfer')
model.summary()

trainable_count = sum([w.numpy().size for w in model.trainable_weights])
print(f"\nTrainable parameters: {trainable_count:,}")

In [ ]:
print("=" * 60)
print("EXPERIMENT 3: Frozen MobileNetV2")
print("=" * 60)

exp3_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='transfer',
    epochs=15,
    learning_rate=1e-4  # Higher LR since only training classification head
)

results_summary.append({
    'architecture': 'transfer (frozen)',
    'learning_rate': 1e-4,
    'epochs': 15,
    'val_auc': exp3_results['results']['auc'],
    'val_acc': exp3_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp3_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp3_results['results']['accuracy']:.1%}")

## Experiment 4: Fine-tuned Transfer Learning

MobileNetV2 with the last 30 layers unfrozen for fine-tuning. This allows the model to adapt high-level ImageNet features to histopathology while keeping low-level features (edges, textures) frozen.

In [ ]:
# Inspect the fine-tuned transfer model
model = get_model('transfer_finetune')

trainable_count = sum([w.numpy().size for w in model.trainable_weights])
total_count = sum([w.numpy().size for w in model.weights])
print(f"Total parameters: {total_count:,}")
print(f"Trainable parameters: {trainable_count:,}")
print(f"Frozen parameters: {total_count - trainable_count:,}")

In [ ]:
print("=" * 60)
print("EXPERIMENT 4: Fine-tuned MobileNetV2")
print("=" * 60)

exp4_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='transfer_finetune',
    epochs=15,
    learning_rate=1e-5  # Lower LR for fine-tuning to avoid catastrophic forgetting
)

results_summary.append({
    'architecture': 'transfer (fine-tuned)',
    'learning_rate': 1e-5,
    'epochs': 15,
    'val_auc': exp4_results['results']['auc'],
    'val_acc': exp4_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp4_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp4_results['results']['accuracy']:.1%}")

## Experiment 5: Simple CNN (sanity check)

The simplest architecture with larger kernels and fewer parameters. If this performs similarly to complex models, the task may be fundamentally limited by the data rather than model capacity.

In [ ]:
print("=" * 60)
print("EXPERIMENT 5: Simple CNN")
print("=" * 60)

exp5_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='simple',
    epochs=15,
    learning_rate=1e-5
)

results_summary.append({
    'architecture': 'simple',
    'learning_rate': 1e-5,
    'epochs': 15,
    'val_auc': exp5_results['results']['auc'],
    'val_acc': exp5_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp5_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp5_results['results']['accuracy']:.1%}")

## Experiment 6: Extended Training

Train the best-performing architecture for more epochs to check if the validation signal strengthens with longer training.

In [ ]:
print("=" * 60)
print("EXPERIMENT 6: Subtle CNN with Extended Training (30 epochs)")
print("=" * 60)

exp6_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='subtle',
    epochs=30,
    learning_rate=1e-5
)

results_summary.append({
    'architecture': 'subtle (30 epochs)',
    'learning_rate': 1e-5,
    'epochs': 30,
    'val_auc': exp6_results['results']['auc'],
    'val_acc': exp6_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp6_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp6_results['results']['accuracy']:.1%}")

## Experiment 7: Higher Learning Rate

Test whether a higher learning rate helps escape local minima.

In [ ]:
print("=" * 60)
print("EXPERIMENT 7: Subtle CNN with Higher LR (1e-4)")
print("=" * 60)

exp7_results = run_binary_experiment(
    dataset_path=TRAIN_PATH,
    experiment_type=3,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-4
)

results_summary.append({
    'architecture': 'subtle (LR=1e-4)',
    'learning_rate': 1e-4,
    'epochs': 15,
    'val_auc': exp7_results['results']['auc'],
    'val_acc': exp7_results['results']['accuracy']
})

print(f"\nValidation AUC: {exp7_results['results']['auc']:.3f}")
print(f"Validation Accuracy: {exp7_results['results']['accuracy']:.1%}")

---

## Results Summary

In [ ]:
# Create summary dataframe
df = pd.DataFrame(results_summary)
df = df.sort_values('val_auc', ascending=False)

print("\n" + "=" * 70)
print("VALIDATION RESULTS SUMMARY (Experiment 3: Slide Context Detection)")
print("=" * 70)
print(df.to_string(index=False))
print("\nNote: Random chance = 0.50 AUC")

In [ ]:
# Visualise results
fig, ax = plt.subplots(figsize=(12, 6))

architectures = df['architecture'].tolist()
aucs = df['val_auc'].tolist()

colours = ['steelblue' if auc > 0.55 else 'gray' for auc in aucs]
bars = ax.barh(architectures, aucs, color=colours)

# Add value labels
for bar, auc in zip(bars, aucs):
    ax.text(auc + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{auc:.3f}', va='center', fontsize=10)

# Add random chance line
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Random chance')

ax.set_xlabel('Validation AUC', fontsize=12)
ax.set_title('Field Cancerisation Detection: Architecture Comparison', fontsize=14, fontweight='bold')
ax.set_xlim(0.4, 0.75)
ax.legend()

plt.tight_layout()
plt.show()

---

## Test Set Evaluation

Evaluate the best-performing model(s) on the held-out test set. This is the true test of generalisation.

In [ ]:
from tensorflow import keras
from src.models import evaluate_on_test_set, load_model_metadata

# Evaluate the baseline subtle model on test set
print("=" * 60)
print("TEST SET EVALUATION: Subtle CNN")
print("=" * 60)

model_path = './models/slide_context_detection.keras'

if Path(model_path).exists():
    model = keras.models.load_model(model_path)
    meta = load_model_metadata(model_path)
    
    test_result = evaluate_on_test_set(
        model, 
        TEST_PATH, 
        {0: ['normal_from_normal'], 1: ['normal_from_tumor']},
        'exp3',
        threshold=meta['threshold'],
        normalise=meta.get('normalise_patches', False)
    )
    
    print(f"\nValidation AUC: {meta.get('val_auc', 'N/A')}")
    print(f"Test AUC: {test_result['auc']:.3f}")
    print(f"Gap: {meta.get('val_auc', 0) - test_result['auc']:.3f}")
    print("\n" + test_result['report'])
else:
    print(f"Model not found at {model_path}")
    print("Run the training cells above to generate the model.")

---

## Analysis and Conclusions

### Key Findings

1. **Validation performance is weak but above chance**: Most architectures achieve validation AUC in the 0.55-0.65 range, suggesting the model detects *something* in the training/validation data.

2. **Test performance is at chance**: When evaluated on held-out test slides, performance drops to ~0.50 AUC (random guessing). The validation signal does not generalise.

3. **Architecture doesn't matter**: Simple CNNs, attention models, and transfer learning all converge to similar weak results. This suggests the limitation is in the data or task, not model capacity.

4. **Extended training doesn't help**: More epochs don't improve performance, ruling out underfitting.

### Interpretation

The validation signal likely reflects **slide-level confounds** rather than genuine field cancerisation:

- **Staining batch effects**: Tumor slides may have been processed differently (different days, reagent batches)
- **Scanner artifacts**: Different scanners or settings between normal and tumor slides
- **Institution differences**: CAMELYON16 data comes from two centres; tumor/normal distribution may correlate with centre

These confounds are shared between training and validation slides (drawn from the same pool) but don't transfer to test slides.

### What Would Be Needed

To properly test the field cancerisation hypothesis:

1. **Multi-instance learning**: Aggregate evidence across many patches per slide, rather than classifying patches independently
2. **Pathology foundation models**: Use feature extractors pre-trained on millions of pathology images (e.g., UNI, CONCH, Phikon) rather than ImageNet or training from scratch
3. **Controlled experiments**: Compare patches from the same slide at different distances from the tumor
4. **Molecular ground truth**: Validate against gene expression or methylation data showing actual field effects

### This is a Valuable Negative Result

Negative results are informative. This experiment establishes:
- Patch-level H&E classification cannot detect field cancerisation in this dataset
- Model capacity is not the bottleneck
- More sophisticated approaches are needed

This guides future work and prevents others from repeating the same experiments.

In [ ]:
# Save results summary to CSV
output_path = './results/field_cancerisation_experiments.csv'
Path('./results').mkdir(exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")